# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use free open source models on Ollama. I also use paid open-source models via Groq and OpenRouter. Only pick the models you want to!
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess


In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key not set (and this is optional)
Grok API Key not set (and this is optional)
Groq API Key not set (and this is optional)
OpenRouter API Key not set (and this is optional)


In [3]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


In [4]:
models = ["qwen3-coder:480b-cloud", "gpt-oss:20b-cloud", "minimax-m2:cloud"]

clients = {"qwen3-coder:480b-cloud": ollama, "gpt-oss:20b-cloud": ollama, "minimax-m2:cloud": ollama}

languages = languages = ["c plus plus", "java", "javascript"]

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [5]:
!ollama pull qwen3-coder:480b-cloud

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 476b4620b85b: 100% ▕██████████████████▏  382 B                         
verifying sha256 digest 
writing manifest 
success 


In [6]:
!ollama pull gpt-oss:20b-cloud

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling cf2ed067e945: 100% ▕██████████████████▏  381 B                         
verifying sha256 digest 
writing manifest 
success 


In [7]:
!ollama pull minimax-m2:cloud

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 32677c818575: 100% ▕██████████████████▏  382 B                         
verifying sha256 digest 
writing manifest 
success 


In [8]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.26200',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i5-8300H CPU @ 2.30GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (x86_64-win32-seh-rev0, Built by MinGW-Builds project) 15.2.0',
   'g++': 'g++.EXE (x86_64-win32-seh-rev0, Built by MinGW-Builds project) 15.2.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

## Overwrite this with the commands from yesterday

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [9]:
compile_command = ["g++.EXE", "-O3", "-std=c++17", "main.cpp", "-o", "main.exe"]
run_command = ["main.exe"]


## And now, on with the main task

In [10]:
system_prompt = """
Your task is to convert Python code into high performance code in the language user ask for.
Respond only with the code in the language user asks for. Do not provide any explanation other than occasional comments.
The converted language code response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python, language):
    # Map language names to file extensions and response formats
    language_configs = {
        "c plus plus": {
            "extension": "cpp",
            "response": "C++",
            "file": "main.cpp",
            "compile": compile_command
        },
        "java": {
            "extension": "java", 
            "response": "Java",
            "file": "Main.java",
            "compile": ["javac", "Main.java"]  # You'll need to set this up
        },
        "javascript": {
            "extension": "js",
            "response": "JavaScript", 
            "file": "main.js",
            "compile": None  # No compilation needed
        }
    }
    
    config = language_configs.get(language, language_configs["c plus plus"])
    
    return f"""
Port this Python code to {config['response']} code with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called {config['file']}
{('The compilation command is: ' + str(config['compile'])) if config['compile'] else 'No compilation needed for JavaScript.'}
Respond only with {config['response']} code.
Python code to port:

```python
{python}
```
"""

In [11]:
def messages_for(python, language):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python, language)}
    ]
 

In [12]:
def write_output(cpp_code, language): 
    language_configs = { 
        "c plus plus": "main.cpp", 
        "java": "Main.java", 
        "javascript": "main.js" 
    }
    filename = language_configs.get(language, "main.cpp")
    
    # Special handling for Java - ensure class name matches filename
    if language == "java":
        # If the code doesn't have a class declaration, wrap it in a Main class
        if "class" not in cpp_code.lower():
            cpp_code = f"""public class Main {{
{cpp_code}
}}"""
        
        # Ensure the class name is exactly "Main"
        cpp_code = cpp_code.replace("class main", "class Main")
        cpp_code = cpp_code.replace("class MAIN", "class Main")
        cpp_code = cpp_code.replace("class Java", "class Main")
        cpp_code = cpp_code.replace("class Program", "class Main")
        
    with open(filename, "w") as f:
        f.write(cpp_code)
    print(f"Code written to {filename}")
    return filename

In [32]:
def clean_code_output(reply, language):
    """Clean the output by removing markdown code fences and unwanted text"""
    # Remove code block markers for different languages
    reply = reply.replace('```cpp', '').replace('```java', '').replace('```js', '').replace('```javascript', '').replace('```', '')
    
    # Split into lines and clean
    lines = reply.split('\n')
    cleaned_lines = []
    
    for line in lines:
        line = line.strip()
        
        # Skip empty lines at the beginning
        if not line and not cleaned_lines:
            continue
            
        # For JavaScript, specifically handle "script" issues
        if language == "javascript":
            # Remove script tags
            line = line.replace('<script>', '').replace('</script>', '')
            
            # Skip lines that are just "script" (case insensitive)
            if line.lower() in ['script', '<script', '</script']:
                continue
                
            # Remove "script" from the beginning of lines
            if line.lower().startswith('script'):
                line = line[6:].strip()  # Remove "script" + space
                if not line:  # If line was just "script", skip it
                    continue
        
        # For Java, ensure class name is Main
        elif language == "java":
            if "class" in line.lower() and "main" not in line.lower():
                # Skip non-Main class declarations for Java
                continue
        
        cleaned_lines.append(line)
    print(cleaned_lines)
    return '\n'.join(cleaned_lines).strip()

In [47]:
def port(model, python, language):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python, language), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    
    # Clean the output based on language
    cleaned_reply = clean_code_output(reply, language)

    # Write to appropriate file based on language
    filename = write_output(cleaned_reply, language)
    print(f"Code written to {filename}")
    return cleaned_reply


In [34]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [48]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [35]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [36]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [43]:
def run_converted_code(language):
    """Run the converted code based on language"""
    if language == "c plus plus":
        try:
            # Check if main.cpp exists
            import os
            if not os.path.exists("main.cpp"):
                return "Error: main.cpp file not found. Please convert Python to C++ first."
            
            # Compile C++
            compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
            if compile_result.returncode != 0:
                return f"Compilation Error:\n{compile_result.stderr}"
            
            # Run C++
            result = subprocess.run(run_command, check=True, text=True, capture_output=True)
            return result.stdout
        except subprocess.CalledProcessError as e:
            return f"Compilation/Execution Error:\n{e.stderr}"
        except Exception as e:
            return f"Error: {e}"
    
    elif language == "java":
        try:
            import os
            
            # Check if Main.java exists
            if not os.path.exists("Main.java"):
                return "Error: Main.java file not found. Please convert Python to Java first."
            
            print("Found Main.java, attempting compilation...")
            
            # Compile Java
            compile_result = subprocess.run(["javac", "Main.java"], check=True, text=True, capture_output=True)
            if compile_result.returncode != 0:
                return f"Compilation Error:\n{compile_result.stderr}"
            
            # Check if Main.class was created
            if not os.path.exists("Main.class"):
                return "Error: Main.class was not created. Check Java code for syntax errors."
            
            # Run Java
            result = subprocess.run(["java", "Main"], check=True, text=True, capture_output=True)
            return result.stdout
        except subprocess.CalledProcessError as e:
            return f"Compilation/Execution Error:\n{e.stderr}"
        except FileNotFoundError:
            return "Error: Java compiler (javac) not found. Please install Java Development Kit (JDK)."
        except Exception as e:
            return f"Error: {e}"
    
    elif language == "javascript":
        try:
            import os
            # Check if main.js exists
            if not os.path.exists("main.js"):
                return "Error: main.js file not found. Please convert Python to JavaScript first."
                
            # For JavaScript, use Node.js if available
            result = subprocess.run(["node", "main.js"], check=True, text=True, capture_output=True)
            return result.stdout
        except subprocess.CalledProcessError as e:
            return f"Execution Error:\n{e.stderr}"
        except FileNotFoundError:
            return "Error: Node.js not found. Please install Node.js to run JavaScript code."
        except Exception as e:
            return f"Error: {e}"
    
    return "Language not supported"

In [39]:
def get_run_button_text(language):
    """Get the text for the run button based on language"""
    return f"Run {language.replace('plus plus', '++').title()}"

In [49]:
from styles import CSS

# Main Gradio Interface with updated layout
with gr.Blocks(css=CSS, title="Code Converter & Runner") as ui:
    
    # Header
    gr.Label("Code Converter & Runner", elem_classes=["py-out"])
    
    # Row 1: Model Selection | Language Selection  
    with gr.Row(elem_classes=["controls"]):
        model = gr.Dropdown(models, label="Select LLM", value=models[0])
        language = gr.Dropdown(languages, label="Select language", value=languages[0])
    
    # Row 2: Python Code | Generated Code
    with gr.Row():
        python = gr.TextArea(label="Python code:", lines=28, value=python_hard, elem_classes=["card"])
        cpp = gr.TextArea(label="Generated code:", lines=28, elem_classes=["card"])
    
    # Row 3: Convert Button
    with gr.Row(elem_classes=["controls"]):
        convert = gr.Button("Convert Code", variant="primary", elem_classes=["convert-btn"])
    
    # Row 4: Run Python | Run Language
    with gr.Row(elem_classes=["controls"]):
        run_python_btn = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        run_converted_btn = gr.Button(get_run_button_text(languages[0]), elem_classes=["run-btn", "cpp"])
    
    # Row 5: Output Results
    with gr.Row():
        python_output = gr.TextArea(label="Python Output:", lines=8, elem_classes=["py-out"])
        converted_output = gr.TextArea(label="Converted Code Output:", lines=8, elem_classes=["cpp-out"])
    
    # Update the run converted button text when language changes
    def update_run_button_text(selected_language):
        return get_run_button_text(selected_language)
    
    language.change(update_run_button_text, inputs=[language], outputs=[run_converted_btn])
    
    # Button click handlers
    convert.click(port, inputs=[model, python, language], outputs=[cpp], show_progress="full")
    
    run_python_btn.click(run_python, inputs=[python], outputs=[python_output], show_progress="full")
    
    run_converted_btn.click(
        lambda lang: run_converted_code(lang), 
        inputs=[language], 
        outputs=[converted_output], 
        show_progress="full"
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


['// Fast JavaScript port of the given Python code', '(function () {', "'use strict';", '', '// Linear Congruential Generator (LCG) - identical to the Python version', 'function lcg(seed, a, c, m) {', 'let value = seed | 0;', 'return function () {', "// All operations performed as signed 32-bit to match Python's int mod 2**32 behavior", 'value = Math.imul(a, value) + c;', '// Keep value in 32-bit range', 'value |= 0;', 'return value >>> 0; // unsigned 32-bit', '};', '}', '', '// Compute the maximum subarray sum using the same O(n^2) algorithm as the Python code', '// to produce identical results.', 'function maxSubarraySum(arr, minVal, maxVal) {', 'const n = arr.length;', "// Initialize with the smallest possible safe integer (same as Python's float('-inf') semantics)", 'let maxSum = -Infinity;', 'let i = 0;', 'for (; i < n; ++i) {', 'let currentSum = 0;', 'let j = i;', 'for (; j < n; ++j) {', 'currentSum += arr[j];', 'if (currentSum > maxSum) maxSum = currentSum;', '}', '}', 'return m

In [32]:
compile_and_run()

Result: 3.141592656089
Execution Time: 0.629831 seconds

Result: 3.141592656089
Execution Time: 0.874401 seconds

Result: 3.141592656089
Execution Time: 0.546585 seconds



minimax-m2:cloud: 0.422168  
qwen3-coder:480b-cloud: 0.629831  
gpt-oss:20b-cloud: 0.416775  





In Ed's experiments, the performance speedups were:

9th place: Qwen 2.5 Coder: Fail  
8th place: OpenAI GPT-OSS 120B: 14X speedup    
7th place: DeepSeek Coder v2: 168X speedup  
6th place: Qwen3 Coder 30B: 168X speedup   
5th place: Claude Sonnet 4.5: 184X speedup   
4th place: GPT-5: 233X speedup  
**3rd place: oss-20B: 238X speedup**  
2nd place: Grok 4: 1060X speedup  
1st place: Gemini 2.5 Pro: 1440X speedup  